## Задание 1: MSA

Для своего анализа я взяла последовательности родопсина 1 (RH1, RHO). Родопсин — это светочувствительный GPCR, который находится в палочках сетчатки глаза. Он отвечает за зрение при низкой освещённости. В анализе использовались последовательности четырёх организмов sp|P08100|OPSD HUMAN Homo sapiens - человек, sp|P02699|OPSD BOVIN	Bos taurus - бык, sp|P06002|OPS1 DROME Drosophila melanogaster	- дрозофила, sp|P35359|OPSD DANRE Danio rerio - зебрафиш.

Выравнивание выполнено программой T-Coffee, версия 11.00. Общий Score = 905, что говорит о достаточно хорошем качестве выравнивания (шкала BAD / AVG / GOOD показывает, что участки преимущественно хорошо выровнены). Хорошо видны красные выскоконсервативные домены, которые скорее всего являются трансмембранными петлями GPCR. 

![Picture1.png](images/Picture1.png)

Ниже представлено получившееся дерево. Как можно видеть, соседними группами оказались человек и корова, что логично, так как это двое млекопитающих. Дерево получилось с мультифуркацией - данио-рерио и дрозофиллу однозначно разрешить не удалось. Однако видно, что длина ветви дрозофиллы значительно больше, что также соотносится с эволюционным расположением данных видов.

![jh.png](images/jh.png)

## Задание 2: MSA в MEGA

Для начала было построено филогенетичское дерево для DNA последовательностей S-белка вируса SARS_CoV 2021 года, секвенированных в России. 
Большинство российских штаммов (образцы 10825-11309) группируются очень плотно, что говорит о высокой генетической близости. Образцы 4568 и 3849 имеют более длинные ветви и образуют отдельную кладу. Возможно это более ранние штаммы вируса, циркулировавшие в России в 2021.

![Screenshot 2026-03-27 at 10.46.41 AM.png](images/qweffq.png)

Далее в анализ были добавлены последовательности S-белка трёх изолятов: PZ155425.1 из Бразилии (Collection Date: 2026-02-09), PX974484.1 из Австралии (Collection Date: 2025-07-05) и NC_045512.2 из Китая (Collection Date: 2019-12). Как можно видеть, более новые последовательности группируются вместе (PZ155425.1 и PX974484.1 ) и сильно отличаются от всех остальных (самая длинная ветвь). Изолят  NC_045512.2 из Китая группируется с последовательностями-outliers 3849 и 4568, что подкрепляет гипотезу о том, что эти последовательности относятся к более ранним штаммам, привезённым на территорию России.

![qe.png](images/qe.png)

## Задание 3: BLAST

### 3.1

Дана последовательность белка EGMQCSCGIDYYTPHEETNNESFVIYMFVVHFIIPLIVIFFCYGQLVFTVKEAAAQQQES
С помощью blastp была найдена данная последовательность в базе данных BLAST. Последовательность является частью белка родопсина Bos taurus (скорее всего это фрагмент Chain A). 

![Screenshot 2026-03-28 at 11.21.35 AM.png](images/qefr.png)


Далее с помощью blastp с поиском nr (non-redundant protein sequences) был найден ортолог данного белка у человека.  В итоге был найден человеческий родопсин. Этот белок отвечает за ночное зрение при низкой освещённости. При дефектах в данном гене у человека возникает врожденная стационарная ночная слепота. Blastp был выьран, так как у нас уже есть белковая последовательность, не нужно переводить из нуклеотидов. База nr была выбрана так как в ней можно искать части белка, а не белки целиком. 

![Screenshot 2026-03-28 at 11.11.46 AM.png](images/qwef.png)

### 3.2

1. Для начала были скачаны транскриптомные данные RNA-seq данных светлячка Photinus pyralis с помощью команды:

fastq-dump --split-files --gzip SRR6345446

В итоге получили 15,475,496 парных ридов

2. Далее был собран транскриптом с помощью rnaSPAdes:

rnaspades.py -1 SRR6345446_1.fastq.gz -2 SRR6345446_2.fastq.gz -o photinus_transcriptome -t 4 -m 8

Всего было собрано 86,168 транскриптов

3. Далее с помощью TransDecoder.LongOrfs был предсказаны открытые рамки считывания

TransDecoder.LongOrfs -t photinus_transcriptome/transcripts.fasta

4. Затем была создана белковая база BLAST на основе аннотированных транскриптомных данных

makeblastdb -in transcripts.fasta.transdecoder_dir/longest_orfs.pep  -dbtype prot -out photinus_proteome -title "Photinus pyralis proteome database"

5. Далее была скачана последовательность люциферазы Luciola cruciata

curl "https://rest.uniprot.org/uniprotkb/P08659.fasta" > luciola_luciferase.fasta

6. В трансриптоме были найдены гомологи люциферазы

blastp -query luciola_luciferase.fasta -db photinus_proteome  -out luciferase_hits.txt -outfmt "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore"  -evalue 1e-10 -max_target_seqs 50

7. Далее были отобраны 10 лучших хитов 

sort -k12 -nr luciferase_hits.txt | head -10 > top10_hits.txt
cut -f2 top10_hits.txt > top10_ids.txt

seqtk subseq transcripts.fasta.transdecoder_dir/longest_orfs.pep top10_ids.txt > top10_luciferase_candidates.fasta
